# Fase 4: Procesamiento y Construcción de la Capa Silver
Transformación, estandarización de nombres, casteo estricto de tipos de datos y tratamiento justificado de valores nulos.

In [37]:
from pathlib import Path
import pandas as pd
from IPython.display import display

In [2]:
# 1. Definición de rutas del Data Lakehouse local
ruta_base = Path("..")
ruta_bronze = ruta_base / "data" / "bronze"
ruta_silver = ruta_base / "data" / "silver"
# Asegurar que la carpeta de destino Silver exista
ruta_silver.mkdir(parents=True, exist_ok=True)

In [3]:
# 2. Carga de las 11 tablas desde Parquet
archivos_parquet = sorted(list(ruta_bronze.glob("*.parquet")))
tablas_bronze = {}
for archivo in archivos_parquet:
    tablas_bronze[archivo.stem] = pd.read_parquet(archivo, engine="pyarrow")
print(f"[OK] Se cargaron exitosamente {len(tablas_bronze)} tablas desde data/bronze/\n")

[OK] Se cargaron exitosamente 11 tablas desde data/bronze/



In [4]:
# Resumen de verificación de carga
for nombre, df in tablas_bronze.items():
    print(f"• {nombre:<22} : {len(df):>4} filas x {len(df.columns):>2} columnas")

• categories             :    8 filas x  4 columnas
• customers              :   91 filas x 11 columnas
• employee-territories   :   49 filas x  2 columnas
• employees              :    9 filas x 18 columnas
• order-details          : 2155 filas x  5 columnas
• orders                 :  830 filas x 14 columnas
• products               :   77 filas x 10 columnas
• regions                :    4 filas x  2 columnas
• shippers               :    3 filas x  3 columnas
• suppliers              :   29 filas x 12 columnas
• territories            :   53 filas x  3 columnas


### 1 - Transformación de Dimensiones Maestras (Bloque 1)

In [5]:
tablas_silver = {}
# Función auxiliar para recortar espacios en blanco en todas las columnas de texto
def limpiar_textos(df):
    df_limpio = df.copy()
    for col in df_limpio.select_dtypes(include=["object", "string"]).columns:
        df_limpio[col] = df_limpio[col].astype(str).str.strip()
    return df_limpio

In [6]:
# A. Categorías
df_cat = limpiar_textos(tablas_bronze["categories"])
tablas_silver["categories"] = df_cat

In [7]:
# B. Regiones
df_reg = limpiar_textos(tablas_bronze["regions"])
tablas_silver["regions"] = df_reg

In [8]:
# C. Territorios
df_ter = limpiar_textos(tablas_bronze["territories"])
tablas_silver["territories"] = df_ter

In [9]:
# D. Transportistas
df_ship = limpiar_textos(tablas_bronze["shippers"])
tablas_silver["shippers"] = df_ship

In [10]:
# E. Empleados-Territorios (Tabla puente)
df_et = limpiar_textos(tablas_bronze["employee-territories"])
tablas_silver["employee-territories"] = df_et

In [11]:
# F. Proveedores (Aplicación de reglas de imputación)
df_sup = limpiar_textos(tablas_bronze["suppliers"])
df_sup["region"] = df_sup["region"].replace({"nan": "Sin Región", "": "Sin Región"}).fillna("Sin Región")
df_sup["fax"] = df_sup["fax"].replace({"nan": "Sin Fax", "": "Sin Fax"}).fillna("Sin Fax")
df_sup["homePage"] = df_sup["homePage"].replace({"nan": "Sin Web", "": "Sin Web"}).fillna("Sin Web")
tablas_silver["suppliers"] = df_sup

In [12]:
# Validación inmediata de nulos en proveedores
print("=== VALIDACIÓN DE LA TABLA SUPPLIERS (SILVER) ===")
print(f"Total filas: {len(df_sup)}")
print(f"Nulos restantes en 'region': {(df_sup['region'] == 'Sin Región').sum()} filas imputadas con 'Sin Región'")
print(f"Valores nulos reales (NaN/None) en suppliers: {df_sup.isna().sum().sum()}")

=== VALIDACIÓN DE LA TABLA SUPPLIERS (SILVER) ===
Total filas: 29
Nulos restantes en 'region': 20 filas imputadas con 'Sin Región'
Valores nulos reales (NaN/None) en suppliers: 0


### 2 - Transformación de Entidades Principales (Bloque 2)

In [13]:
# A. CLIENTES (customers)
df_cust = limpiar_textos(tablas_bronze["customers"])
df_cust["region"] = df_cust["region"].replace({"nan": "Sin Región", "": "Sin Región"}).fillna("Sin Región")
df_cust["postalCode"] = df_cust["postalCode"].replace({"nan": "Sin Código", "": "Sin Código"}).fillna("Sin Código")
df_cust["fax"] = df_cust["fax"].replace({"nan": "Sin Fax", "": "Sin Fax"}).fillna("Sin Fax")
tablas_silver["customers"] = df_cust

In [14]:
# B. EMPLEADOS (employees)
df_emp = limpiar_textos(tablas_bronze["employees"])

In [15]:
# Casteo estricto de fechas
df_emp["birthDate"] = pd.to_datetime(df_emp["birthDate"])
df_emp["hireDate"] = pd.to_datetime(df_emp["hireDate"])

In [ ]:
# Imputación de región
df_emp["region"] = df_emp["region"].replace({"nan": "Sin Región", "": "Sin Región"}).fillna("Sin Región")

In [17]:
# Casteo de reportsTo a entero nullable (Int64)
df_emp["reportsTo"] = pd.to_numeric(df_emp["reportsTo"], errors="coerce").astype("Int64")

In [18]:
# Enriquecimiento para BI: Nombre completo del empleado
df_emp["fullName"] = df_emp["firstName"] + " " + df_emp["lastName"]
tablas_silver["employees"] = df_emp

In [19]:
# C. PRODUCTOS (products)
df_prod = limpiar_textos(tablas_bronze["products"])

In [20]:
# Asegurar tipos numéricos y redondeos de dinero
df_prod["unitPrice"] = df_prod["unitPrice"].astype(float).round(2)
df_prod["unitsInStock"] = df_prod["unitsInStock"].astype(int)
df_prod["unitsOnOrder"] = df_prod["unitsOnOrder"].astype(int)
df_prod["reorderLevel"] = df_prod["reorderLevel"].astype(int)

In [21]:
# Enriquecimiento para BI: Estado descriptivo del producto
df_prod["discontinuedStatus"] = df_prod["discontinued"].apply(
    lambda x: "Descontinuado" if x == 1 else "Activo"
)
tablas_silver["products"] = df_prod

In [22]:
# VALIDACIONES TÉCNICAS
print("=== VALIDACIÓN TÉCNICA DEL BLOQUE 2 ===")
print(f"1. Customers  -> Filas: {len(df_cust)} | Nulos reales: {df_cust.isna().sum().sum()}")
print(f"2. Employees  -> Filas: {len(df_emp)}  | Tipo birthDate: {df_emp['birthDate'].dtype} | Tipo hireDate: {df_emp['hireDate'].dtype}")
print(f"3. Products   -> Filas: {len(df_prod)} | Tipo unitPrice: {df_prod['unitPrice'].dtype}")
print(f"\nDistribución de productos por estado:")
print(df_prod["discontinuedStatus"].value_counts())

=== VALIDACIÓN TÉCNICA DEL BLOQUE 2 ===
1. Customers  -> Filas: 91 | Nulos reales: 0
2. Employees  -> Filas: 9  | Tipo birthDate: datetime64[us] | Tipo hireDate: datetime64[us]
3. Products   -> Filas: 77 | Tipo unitPrice: float64

Distribución de productos por estado:
discontinuedStatus
Activo           69
Descontinuado     8
Name: count, dtype: int64


### 3 - Transformación Transaccional (Bloque 3)

In [23]:
# A. CABECERA DE ÓRDENES (orders)
df_ord = limpiar_textos(tablas_bronze["orders"])

In [24]:
# Casteo estricto de fechas
df_ord["orderDate"] = pd.to_datetime(df_ord["orderDate"])
df_ord["requiredDate"] = pd.to_datetime(df_ord["requiredDate"])
df_ord["shippedDate"] = pd.to_datetime(df_ord["shippedDate"])  # Respeta los 21 nulos como NaT

In [25]:
# Imputación de nulos según especificación docs/reglas_calidad.md
df_ord["shipRegion"] = df_ord["shipRegion"].replace({"nan": "Sin Región", "": "Sin Región"}).fillna("Sin Región")
df_ord["shipPostalCode"] = df_ord["shipPostalCode"].replace({"nan": "Sin Código", "": "Sin Código"}).fillna("Sin Código")

In [26]:
# Asegurar tipo monetario de flete
df_ord["freight"] = df_ord["freight"].astype(float).round(2)

In [27]:
# Enriquecimiento para BI: Tiempo real de despacho en días
df_ord["diasDespacho"] = (df_ord["shippedDate"] - df_ord["orderDate"]).dt.days

In [28]:
# Enriquecimiento para BI: Clasificación del estado de entrega
def clasificar_envio(row):
    if pd.isna(row["shippedDate"]):
        return "Pendiente"
    elif row["shippedDate"] > row["requiredDate"]:
        return "Demorado"
    else:
        return "A Tiempo"
df_ord["estadoEnvio"] = df_ord.apply(clasificar_envio, axis=1)
tablas_silver["orders"] = df_ord

In [29]:
# B. LÍNEAS DE DETALLE (order-details)
df_det = tablas_bronze["order-details"].copy()

In [30]:
# Tipado numérico
df_det["unitPrice"] = df_det["unitPrice"].astype(float).round(2)
df_det["quantity"] = df_det["quantity"].astype(int)
df_det["discount"] = df_det["discount"].astype(float).round(4)

In [32]:
# Métricas financieras a nivel de línea de venta
df_det["importeBruto"] = (df_det["unitPrice"] * df_det["quantity"]).round(2)
df_det["importeDescuento"] = (df_det["unitPrice"] * df_det["quantity"] * df_det["discount"]).round(2)
df_det["importeNeto"] = (df_det["importeBruto"] - df_det["importeDescuento"]).round(2)
tablas_silver["order-details"] = df_det

In [33]:
# VALIDACIONES TÉCNICAS Y TOTALES FINANCIEROS
print("=== VALIDACIÓN TÉCNICA DEL BLOQUE 3 ===")
print(f"1. Orders        -> Filas: {len(df_ord)} | Pedidos con fecha de envío NaT: {df_ord['shippedDate'].isna().sum()}")
print(f"2. Order-Details -> Filas: {len(df_det)} | Importe neto calculado correctamente.")
print("\nDistribución de pedidos por Estado de Envío:")
print(df_ord["estadoEnvio"].value_counts())
print("\n=== TOTALES FINANCIEROS HISTÓRICOS (NORTHWIND SILVER) ===")
print(f"Venta Bruta Total     : ${df_det['importeBruto'].sum():,.2f}")
print(f"Descuentos Otorgados  : ${df_det['importeDescuento'].sum():,.2f}")
print(f"Venta Neta Total      : ${df_det['importeNeto'].sum():,.2f}")

=== VALIDACIÓN TÉCNICA DEL BLOQUE 3 ===
1. Orders        -> Filas: 830 | Pedidos con fecha de envío NaT: 21
2. Order-Details -> Filas: 2155 | Importe neto calculado correctamente.

Distribución de pedidos por Estado de Envío:
estadoEnvio
A Tiempo     772
Demorado      37
Pendiente     21
Name: count, dtype: int64

=== TOTALES FINANCIEROS HISTÓRICOS (NORTHWIND SILVER) ===
Venta Bruta Total     : $1,354,458.59
Descuentos Otorgados  : $88,665.57
Venta Neta Total      : $1,265,793.02


In [35]:
tablas_silver.keys()

dict_keys(['categories', 'regions', 'territories', 'shippers', 'employee-territories', 'suppliers', 'customers', 'employees', 'products', 'orders', 'order-details'])

In [36]:
for nombre in tablas_silver:
    print(nombre)

categories
regions
territories
shippers
employee-territories
suppliers
customers
employees
products
orders
order-details


### 4 - Exportación Física a la Capa Silver (Parquet)

In [38]:
resumen_exportacion = []
for nombre_tabla, df in tablas_silver.items():
    ruta_archivo = ruta_silver / f"{nombre_tabla}.parquet"
    
    # Exportar con compresión Snappy sin índice
    df.to_parquet(ruta_archivo, engine="pyarrow", compression="snappy", index=False)
    
    # Releer para verificar integridad física en disco
    df_verificado = pd.read_parquet(ruta_archivo, engine="pyarrow")
    
    resumen_exportacion.append({
        "tabla": nombre_tabla,
        "filas_silver": len(df_verificado),
        "columnas_silver": len(df_verificado.columns),
        "tamano_kb": round(ruta_archivo.stat().st_size / 1024, 2),
        "estado": "[OK] Exportado con éxito" if len(df_verificado) == len(df) else "[ERROR] Error de filas"
    })
df_reporte_silver = pd.DataFrame(resumen_exportacion)
print("=" * 70)
print(" REPORTE DE EXPORTACIÓN A CAPA SILVER (data/silver/)")
print("=" * 70)
display(df_reporte_silver)


 REPORTE DE EXPORTACIÓN A CAPA SILVER (data/silver/)


,tabla,filas_silver,columnas_silver,tamano_kb,estado
0,categories,8,4,4.45,[OK] Exportado con éxito
1,regions,4,2,1.83,[OK] Exportado con éxito
2,territories,53,3,3.29,[OK] Exportado con éxito
3,shippers,3,3,2.35,[OK] Exportado con éxito
4,employee-territories,49,2,2.12,[OK] Exportado con éxito
5,suppliers,29,12,10.74,[OK] Exportado con éxito
6,customers,91,11,16.46,[OK] Exportado con éxito
7,employees,9,19,15.51,[OK] Exportado con éxito
8,products,77,11,10.61,[OK] Exportado con éxito
9,orders,830,16,42.55,[OK] Exportado con éxito


In [40]:
# Validación final
total_errores = (df_reporte_silver["estado"] != "[OK] Exportado con éxito").sum()
if total_errores == 0:
    print(f"\nLas 11 tablas de la Capa Silver han sido materializadas en Parquet.")
else:
    print(f"\nHubo errores en {total_errores} tablas.")


Las 11 tablas de la Capa Silver han sido materializadas en Parquet.
